# Prediction Model - LOB

Goal: predict `Lob Associata` (LOB) and evaluate whether the task can be automated.

Workflow:
1. Data checks (3 key questions)
2. Fast manual prefix analysis
3. Rule baseline (no ML)
4. ML baselines (Decision Tree, Naive Bayes, KNN)
5. Interpretable rules and automation/manual-review policy

## 1) Setup

In [30]:
import re
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report, top_k_accuracy_score
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

## 2) Load Data

In [31]:
NOTEBOOK_DIR = Path.cwd()
CANDIDATES = [
    NOTEBOOK_DIR / 'DB for DTM Project.xlsx',
    NOTEBOOK_DIR.parent / 'DB for DTM Project.xlsx',
]
FILE_PATH = next((p for p in CANDIDATES if p.exists()), None)
if FILE_PATH is None:
    raise FileNotFoundError('DB for DTM Project.xlsx not found in current or parent folder')

SHEET = 'Associazioni Cod. Art. - LOB'
df = pd.read_excel(FILE_PATH, sheet_name=SHEET)
df.columns = [str(c).strip() for c in df.columns]

print('File:', FILE_PATH)
print('Shape:', df.shape)
print('Columns:', list(df.columns))

df.head(3)

File: c:\Users\sveta\Documents\vem-product\DB for DTM Project.xlsx
Shape: (22655, 5)
Columns: ['Codice Articolo', 'Lob Associata', 'Nome LOB', 'Inventario', 'Descrizione Articolo']


,Codice Articolo,Lob Associata,Nome LOB,Inventario,Descrizione Articolo
0,000050,1002,CABLAGGIO ALTERNATIVO,Inventario,PATCH CORD UTP 2/RJ45 CAT5E MT2
1,000081,1002,CABLAGGIO ALTERNATIVO,Inventario,cab - cf54/150ez passerella a filo 3m
2,‎000411AA230621QMANE,3004,CLOUD VARIE HW,Inventario,QUARKZMAN 30pz Ultra Sottile Telefono Leva Ape...


## 3) Define Core Columns

In [32]:
CODE_COL = 'Codice Articolo'
LOB_COL = 'Lob Associata'
LOB_NAME_COL = 'Nome LOB'
INV_COL = 'Inventario'
DESC_COL = 'Descrizione Articolo'

required = [CODE_COL, LOB_COL, INV_COL, DESC_COL]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

df_work = df[[c for c in [CODE_COL, LOB_COL, LOB_NAME_COL, INV_COL, DESC_COL] if c in df.columns]].copy()
for c in [CODE_COL, LOB_COL, INV_COL, DESC_COL]:
    df_work[c] = df_work[c].fillna('')

# Optional: apply cleaned LOB labels from data_quality output if available
DATA_QUALITY_DIR = NOTEBOOK_DIR / 'data_quality'
cleaned_target_path = DATA_QUALITY_DIR / 'associazioni_with_lob_cleaned.csv'

def _norm_lob(x):
    s = '' if pd.isna(x) else str(x).strip()
    if s.endswith('.0') and s[:-2].isdigit():
        s = s[:-2]
    return s

if cleaned_target_path.exists():
    clean_cols = [CODE_COL, 'Lob Associata Cleaned']
    df_clean = pd.read_csv(cleaned_target_path, usecols=[c for c in clean_cols if c in pd.read_csv(cleaned_target_path, nrows=0).columns])
    if CODE_COL in df_clean.columns and 'Lob Associata Cleaned' in df_clean.columns:
        df_clean['Lob Associata Cleaned'] = df_clean['Lob Associata Cleaned'].apply(_norm_lob)
        before = df_work[LOB_COL].astype(str).apply(_norm_lob)
        df_work = df_work.merge(df_clean[[CODE_COL, 'Lob Associata Cleaned']], on=CODE_COL, how='left')
        df_work[LOB_COL] = df_work['Lob Associata Cleaned'].fillna(df_work[LOB_COL]).astype(str).apply(_norm_lob)
        changed = int((before != df_work[LOB_COL]).sum())
        print(f'Applied cleaned LOB labels from {cleaned_target_path.name}: changed rows = {changed}')
    else:
        print('Cleaned target file found, but required columns are missing; using original LOB labels.')
else:
    print('Cleaned target file not found; using original LOB labels.')

print('Working shape:', df_work.shape)
df_work.head(3)



Applied cleaned LOB labels from associazioni_with_lob_cleaned.csv: changed rows = 53
Working shape: (22655, 6)


,Codice Articolo,Lob Associata,Nome LOB,Inventario,Descrizione Articolo,Lob Associata Cleaned
0,000050,1002,CABLAGGIO ALTERNATIVO,Inventario,PATCH CORD UTP 2/RJ45 CAT5E MT2,1002
1,000081,1002,CABLAGGIO ALTERNATIVO,Inventario,cab - cf54/150ez passerella a filo 3m,1002
2,‎000411AA230621QMANE,3004,CLOUD VARIE HW,Inventario,QUARKZMAN 30pz Ultra Sottile Telefono Leva Ape...,3004


## 4) Step 1 - Data Checks (3 Questions)

Q1. One codice -> always one LOB?

Q2. Are there clear code patterns?

Q3. Is there a relationship between LOB and Inventario?

In [33]:
# Q1: One codice -> one LOB?
code_lob_uniques = (
    df_work.groupby(CODE_COL)[LOB_COL]
    .nunique(dropna=True)
    .reset_index(name='lob_nunique')
)

ambiguous_codes = code_lob_uniques[code_lob_uniques['lob_nunique'] > 1].copy()

print('Unique codici:', len(code_lob_uniques))
print('Codici with >1 LOB:', len(ambiguous_codes))
print('Share ambiguous:', f"{len(ambiguous_codes)/len(code_lob_uniques):.2%}")

ambiguous_codes.head(20)

Unique codici: 22655
Codici with >1 LOB: 0
Share ambiguous: 0.00%


,Codice Articolo,lob_nunique


In [34]:
# Helper prefix extractor for manual analysis
def extract_prefix(code: str, n: int = 5) -> str:
    text = str(code).upper().strip()
    text = re.sub(r'[^A-Z0-9]', '', text)
    if not text:
        return 'EMPTY'
    return text[:n]

df_work['prefix'] = df_work[CODE_COL].apply(extract_prefix)

# Q2: clear patterns in code prefixes?
prefix_lob = (
    df_work.groupby(['prefix', LOB_COL])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)

prefix_stats = prefix_lob.copy()
prefix_stats['prefix_total'] = prefix_stats.groupby('prefix')['count'].transform('sum')
prefix_stats['confidence'] = prefix_stats['count'] / prefix_stats['prefix_total']

prefix_top = (
    prefix_stats.sort_values(['prefix', 'confidence', 'count'], ascending=[True, False, False])
    .drop_duplicates('prefix')
    .rename(columns={LOB_COL: 'dominant_lob', 'count': 'dominant_count'})
    .sort_values(['confidence', 'prefix_total'], ascending=False)
)

print('Prefixes:', prefix_top.shape[0])
print('Strong prefixes (support>=20 and confidence>=0.9):', int(((prefix_top['prefix_total'] >= 20) & (prefix_top['confidence'] >= 0.9)).sum()))

prefix_top.head(30)

Prefixes: 10125
Strong prefixes (support>=20 and confidence>=0.9): 43


,prefix,dominant_lob,dominant_count,prefix_total,confidence
4317,CPAPS,6002,102,102,1.0
5405,FAS27,3014,43,43,1.0
2557,A1200,1006,36,36,1.0
2704,ACDC2,17001,33,33,1.0
4609,CVN07,1003,28,28,1.0
3130,ASA55,6001,27,27,1.0
4731,DG7GM,3002,27,27,1.0
4972,E3NMX,2016,27,27,1.0
9998,SWONT,3014,27,27,1.0
11100,WSC35,2002,26,26,1.0


In [35]:
# Q3: relationship LOB <-> Inventario
ct_count = pd.crosstab(df_work[LOB_COL], df_work[INV_COL])
ct_row_pct = pd.crosstab(df_work[LOB_COL], df_work[INV_COL], normalize='index').round(3)

print('LOB x Inventario (counts):')
display(ct_count.head(20))

print('LOB x Inventario (row percentages):')
display(ct_row_pct.head(20))

LOB x Inventario (counts):


Inventario,Assistenza,Inventario,Non in inventario
Lob Associata,,,
1,0,1,0
1001,0,498,0
1002,0,2014,3
1003,0,1068,11
1006,0,444,0
1008,0,418,0
1011,0,34,0
1015,0,44,4
1016,0,19,0


LOB x Inventario (row percentages):


Inventario,Assistenza,Inventario,Non in inventario
Lob Associata,,,
1,0.000,1.000,0.000
1001,0.000,1.000,0.000
1002,0.000,0.999,0.001
1003,0.000,0.990,0.010
1006,0.000,1.000,0.000
1008,0.000,1.000,0.000
1011,0.000,1.000,0.000
1015,0.000,0.917,0.083
1016,0.000,1.000,0.000


## 5) Step 2 - Fast Manual Prefix Analysis

Simple table: `prefix -> dominant LOB` with support and confidence.

In [36]:
manual_prefix_table = prefix_top[['prefix', 'dominant_lob', 'prefix_total', 'dominant_count', 'confidence']].copy()
manual_prefix_table = manual_prefix_table.sort_values(['confidence', 'prefix_total'], ascending=False)

manual_prefix_table.head(50)

,prefix,dominant_lob,prefix_total,dominant_count,confidence
4317,CPAPS,6002,102,102,1.0
5405,FAS27,3014,43,43,1.0
2557,A1200,1006,36,36,1.0
2704,ACDC2,17001,33,33,1.0
4609,CVN07,1003,28,28,1.0
3130,ASA55,6001,27,27,1.0
4731,DG7GM,3002,27,27,1.0
4972,E3NMX,2016,27,27,1.0
9998,SWONT,3014,27,27,1.0
11100,WSC35,2002,26,26,1.0


## 6) Step 3 - Rule Baseline (No ML)

Rule: `if prefix = X -> LOB = Y` (learned on train only).

In [37]:
rule_df = df_work[[CODE_COL, LOB_COL, 'prefix']].copy()
rule_df = rule_df[rule_df[LOB_COL].astype(str).str.strip() != '']

# Stratified split requires at least 2 samples per class.
lob_counts = rule_df[LOB_COL].value_counts()
rare_lobs = set(lob_counts[lob_counts < 2].index)

rule_df_model = rule_df[~rule_df[LOB_COL].isin(rare_lobs)].copy()
rule_df_rare = rule_df[rule_df[LOB_COL].isin(rare_lobs)].copy()

print('Total rows for rule baseline:', len(rule_df))
print('Rows excluded as rare LOB (<2 samples):', len(rule_df_rare))
print('Rare LOB classes excluded:', len(rare_lobs))

if rule_df_model[LOB_COL].nunique() < 2:
    raise ValueError('Not enough non-rare classes for stratified split in rule baseline.')

train_df, test_df = train_test_split(
    rule_df_model,
    test_size=0.2,
    random_state=42,
    stratify=rule_df_model[LOB_COL]
)

prefix_to_lob = (
    train_df.groupby('prefix')[LOB_COL]
    .agg(lambda s: s.value_counts().idxmax())
    .to_dict()
)

default_lob = train_df[LOB_COL].value_counts().idxmax()

test_pred_rule = test_df['prefix'].map(prefix_to_lob).fillna(default_lob)

rule_acc = accuracy_score(test_df[LOB_COL], test_pred_rule)
rule_f1_macro = f1_score(test_df[LOB_COL], test_pred_rule, average='macro')
rule_f1_weighted = f1_score(test_df[LOB_COL], test_pred_rule, average='weighted')

rule_metrics = pd.DataFrame([
    {'model': 'Prefix Rule Baseline', 'accuracy': rule_acc, 'f1_macro': rule_f1_macro, 'f1_weighted': rule_f1_weighted}
])
rule_metrics



Total rows for rule baseline: 22655
Rows excluded as rare LOB (<2 samples): 13
Rare LOB classes excluded: 13


,model,accuracy,f1_macro,f1_weighted
0,Prefix Rule Baseline,0.556193,0.441478,0.576102


## 7) Step 4 - ML Baselines for LOB

Models:
- Decision Tree (mandatory)
- Naive Bayes (text-based)
- KNN (optional, included here)

In [38]:
ml_df = df_work[[CODE_COL, DESC_COL, INV_COL, LOB_COL, 'prefix']].copy()
ml_df = ml_df[ml_df[LOB_COL].astype(str).str.strip() != '']
ml_df[DESC_COL] = ml_df[DESC_COL].fillna('')

# Reuse rare classes from previous step if available; otherwise compute here.
if 'rare_lobs' not in globals():
    _counts = ml_df[LOB_COL].value_counts()
    rare_lobs = set(_counts[_counts < 2].index)

ml_df_model = ml_df[~ml_df[LOB_COL].isin(rare_lobs)].copy()

# Text features without target leakage
ml_df_model['input_text'] = (
    'CODE_' + ml_df_model[CODE_COL].astype(str) + ' ' +
    'PREFIX_' + ml_df_model['prefix'].astype(str) + ' ' +
    'INV_' + ml_df_model[INV_COL].astype(str) + ' ' +
    ml_df_model[DESC_COL].astype(str)
)

X = ml_df_model['input_text']
y = ml_df_model[LOB_COL].astype(str)

if y.nunique() < 2:
    raise ValueError('Not enough non-rare classes for ML modeling.')

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

vectorizer = CountVectorizer(lowercase=True, min_df=5, max_features=6000, ngram_range=(1,2), binary=True)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
feature_names = vectorizer.get_feature_names_out()

print('Rows excluded as rare LOB (<2 samples):', len(ml_df) - len(ml_df_model))
print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Classes (after rare-class exclusion):', y.nunique())



Rows excluded as rare LOB (<2 samples): 13
Train shape: (18113, 6000)
Test shape: (4529, 6000)
Classes (after rare-class exclusion): 104


In [39]:
models = {
    'Decision Tree': DecisionTreeClassifier(
        random_state=42,
        max_depth=35,
        min_samples_leaf=5,
        class_weight='balanced'
    ),
    'Naive Bayes': MultinomialNB(alpha=0.5),
    'KNN': KNeighborsClassifier(
        n_neighbors=7,
        weights='distance',
        metric='cosine',
        algorithm='brute'
    )
}

results = []
trained_models = {}
predictions = {}
probabilities = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    proba = model.predict_proba(X_test)

    results.append({
        'model': name,
        'accuracy': accuracy_score(y_test, pred),
        'f1_macro': f1_score(y_test, pred, average='macro'),
        'f1_weighted': f1_score(y_test, pred, average='weighted')
    })

    trained_models[name] = model
    predictions[name] = pred
    probabilities[name] = proba

ml_metrics = pd.DataFrame(results).sort_values('f1_macro', ascending=False).reset_index(drop=True)

compare_df = pd.concat([rule_metrics, ml_metrics], ignore_index=True).sort_values('f1_macro', ascending=False).reset_index(drop=True)
compare_df

,model,accuracy,f1_macro,f1_weighted
0,KNN,0.649371,0.463693,0.644658
1,Prefix Rule Baseline,0.556193,0.441478,0.576102
2,Naive Bayes,0.654228,0.372531,0.634601
3,Decision Tree,0.177081,0.281057,0.186321


In [40]:
for name in ml_metrics['model']:
    print(f'\n===== {name} =====')
    print(classification_report(y_test, predictions[name], zero_division=0))


===== KNN =====
              precision    recall  f1-score   support

        1001       0.67      0.62      0.64       100
        1002       0.52      0.68      0.59       403
        1003       0.44      0.54      0.49       216
        1006       0.70      0.67      0.69        89
        1008       0.82      0.71      0.76        84
        1011       0.00      0.00      0.00         7
        1015       0.00      0.00      0.00        10
        1016       0.00      0.00      0.00         4
       15007       0.77      0.71      0.74        14
       15008       0.71      0.77      0.74        13
       17001       0.69      0.69      0.69       202
       17005       0.50      0.33      0.40        36
       17007       0.00      0.00      0.00         5
       17008       1.00      0.33      0.50         3
       17009       0.44      0.55      0.49        33
       17999       0.00      0.00      0.00         2
       19001       0.00      0.00      0.00         2
       200

## 8) Interpretable Patterns / Rules

### 8.1 Prefix Rules (human-readable)

In [41]:
interpretable_prefix_rules = manual_prefix_table[
    (manual_prefix_table['prefix_total'] >= 20) &
    (manual_prefix_table['confidence'] >= 0.90)
].copy()

interpretable_prefix_rules.head(100)

,prefix,dominant_lob,prefix_total,dominant_count,confidence
4317,CPAPS,6002,102,102,1.000000
5405,FAS27,3014,43,43,1.000000
2557,A1200,1006,36,36,1.000000
2704,ACDC2,17001,33,33,1.000000
4609,CVN07,1003,28,28,1.000000
3130,ASA55,6001,27,27,1.000000
4731,DG7GM,3002,27,27,1.000000
4972,E3NMX,2016,27,27,1.000000
9998,SWONT,3014,27,27,1.000000
11100,WSC35,2002,26,26,1.000000


### 8.2 Decision Tree rules and top features

In [42]:
dt = trained_models['Decision Tree']

imp = pd.DataFrame({
    'feature': feature_names,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

print('Top feature importances:')
display(imp.head(30))

print('\nDecision Tree rules (truncated):')
print(export_text(dt, feature_names=list(feature_names), max_depth=4))

Top feature importances:


,feature,importance
4939,red hat,0.025918
1533,code_a,0.023700
5401,startup,0.023305
1889,copertura,0.023096
3205,inventario cyber,0.022941
1738,code_rs,0.022428
1709,code_pan,0.022398
3565,meraki,0.021715
1473,citrix,0.021259
1657,code_hx,0.019028



Decision Tree rules (truncated):
|--- red hat <= 0.50
|   |--- code_a <= 0.50
|   |   |--- startup <= 0.50
|   |   |   |--- copertura <= 0.50
|   |   |   |   |--- inventario cyber <= 0.50
|   |   |   |   |   |--- truncated branch of depth 31
|   |   |   |   |--- inventario cyber >  0.50
|   |   |   |   |   |--- truncated branch of depth 9
|   |   |   |--- copertura >  0.50
|   |   |   |   |--- cisco <= 0.50
|   |   |   |   |   |--- class: 17009
|   |   |   |   |--- cisco >  0.50
|   |   |   |   |   |--- class: 99005
|   |   |--- startup >  0.50
|   |   |   |--- prefix_wstrt <= 0.50
|   |   |   |   |--- service <= 0.50
|   |   |   |   |   |--- class: 20011
|   |   |   |   |--- service >  0.50
|   |   |   |   |   |--- class: 99004
|   |   |   |--- prefix_wstrt >  0.50
|   |   |   |   |--- class: 17008
|   |--- code_a >  0.50
|   |   |--- premises <= 0.50
|   |   |   |--- smart <= 0.50
|   |   |   |   |--- meetings <= 0.50
|   |   |   |   |   |--- truncated branch of depth 11
|   |   |  

### 8.3 Naive Bayes class tokens

In [43]:
nb = trained_models['Naive Bayes']
nb_rows = []
for class_idx, cls in enumerate(nb.classes_):
    top_idx = np.argsort(nb.feature_log_prob_[class_idx])[::-1][:20]
    for rank, idx in enumerate(top_idx, start=1):
        nb_rows.append({
            'lob': cls,
            'rank': rank,
            'token': feature_names[idx],
            'log_prob': nb.feature_log_prob_[class_idx, idx]
        })

nb_tokens_df = pd.DataFrame(nb_rows)
nb_tokens_df

,lob,rank,token,log_prob
0,1001,1,inv_inventario,-2.896764
1,1001,2,lszh,-4.435955
2,1001,3,cat,-4.521373
3,1001,4,rj45,-4.600885
4,1001,5,utp,-4.798495
...,...,...,...,...
2075,99999,16,ev,-7.633047
2076,99999,17,24v,-7.633047
2077,99999,18,auto,-7.633047
2078,99999,19,code_prestazione,-7.633047


## 9) Automation vs Manual Review

In [44]:
best_model_name = ml_metrics.iloc[0]['model']
best_model = trained_models[best_model_name]
best_pred = predictions[best_model_name]
best_proba = probabilities[best_model_name]

max_proba = best_proba.max(axis=1)
conf_bucket = pd.cut(max_proba, bins=[-1, 0.65, 0.85, 1.0], labels=['low_manual_review', 'medium', 'high'])

policy_df = pd.DataFrame({
    'y_true': y_test.reset_index(drop=True),
    'y_pred': pd.Series(best_pred),
    'max_proba': max_proba,
    'bucket': conf_bucket
})
policy_df['correct'] = (policy_df['y_true'] == policy_df['y_pred']).astype(int)

bucket_summary = policy_df.groupby('bucket', observed=False).agg(
    n=('correct', 'size'),
    accuracy=('correct', 'mean')
).reset_index()
bucket_summary['share'] = bucket_summary['n'] / len(policy_df)

# Rule-based auto-cases
strong_prefixes = set(interpretable_prefix_rules['prefix']) if 'interpretable_prefix_rules' in globals() else set()
rule_auto_share = test_df['prefix'].isin(strong_prefixes).mean() if len(test_df) else 0.0

print('Best ML model:', best_model_name)
print('Share of strong prefix-rule cases in test:', f'{rule_auto_share:.2%}')

display(bucket_summary)

Best ML model: KNN
Share of strong prefix-rule cases in test: 7.02%


,bucket,n,accuracy,share
0,low_manual_review,2013,0.419771,0.444469
1,medium,758,0.726913,0.167366
2,high,1758,0.878840,0.388165


## 10) Final Conclusion Template

Fill from outputs above:
1. Can we automate LOB prediction?
2. Where is model/rule confidence high?
3. Where is manual review needed?

Suggested policy:
- Auto-assign when strong prefix rule exists OR ML confidence is high.
- Manual review for low-confidence and rare/ambiguous cases.

In [45]:
best = compare_df.iloc[0]
print('FINAL SUMMARY - LOB')
print(f"Best approach: {best['model']}")
print(f"Accuracy: {best['accuracy']:.4f}")
print(f"F1 macro: {best['f1_macro']:.4f}")
print(f"F1 weighted: {best['f1_weighted']:.4f}")
print(f"Rare LOB classes excluded from train/test: {len(rare_lobs)}")

if 'bucket_summary' in globals() and not bucket_summary.empty:
    high = bucket_summary[bucket_summary['bucket'] == 'high']
    low = bucket_summary[bucket_summary['bucket'] == 'low_manual_review']
    if len(high):
        print(f"High-confidence share: {high['share'].iloc[0]:.2%}, accuracy: {high['accuracy'].iloc[0]:.2%}")
    if len(low):
        print(f"Manual-review share (low confidence): {low['share'].iloc[0]:.2%}")

print('Use interpretable artifacts:')
print('- Prefix rules: interpretable_prefix_rules')
print('- Decision Tree rules: export_text output')
print('- Naive Bayes tokens: nb_tokens_df')


FINAL SUMMARY - LOB
Best approach: KNN
Accuracy: 0.6494
F1 macro: 0.4637
F1 weighted: 0.6447
Rare LOB classes excluded from train/test: 13
High-confidence share: 38.82%, accuracy: 87.88%
Manual-review share (low confidence): 44.45%
Use interpretable artifacts:
- Prefix rules: interpretable_prefix_rules
- Decision Tree rules: export_text output
- Naive Bayes tokens: nb_tokens_df


## 11) Hierarchical Baseline (Group -> Detailed LOB)

Two-stage prediction:
1. predict 2-digit LOB group,
2. predict detailed LOB inside predicted group.


In [46]:

def lob_group(code: str) -> str:
    s = str(code).strip()
    if s.endswith('.0') and s[:-2].isdigit():
        s = s[:-2]
    digits = ''.join(ch for ch in s if ch.isdigit())
    if not digits:
        return 'NA'
    return digits.zfill(2)[:2]

# Use same train/test split and vectorized matrices from ML baseline section
y_train_reset = y_train.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)
y_train_group = y_train_reset.astype(str).apply(lob_group)
y_test_group = y_test_reset.astype(str).apply(lob_group)

# Stage 1: group model
group_model = MultinomialNB(alpha=0.5)
group_model.fit(X_train, y_train_group)
group_pred = group_model.predict(X_test)

# Stage 2: one model per group for detailed class
group_detail_models = {}
global_default_lob = y_train_reset.value_counts().idxmax()

for g in sorted(y_train_group.unique()):
    idx = np.where(y_train_group.values == g)[0]
    y_sub = y_train_reset.iloc[idx]
    X_sub = X_train[idx]

    if y_sub.nunique() == 1:
        group_detail_models[g] = {'type': 'constant', 'label': y_sub.iloc[0]}
    else:
        m = MultinomialNB(alpha=0.5)
        m.fit(X_sub, y_sub)
        group_detail_models[g] = {'type': 'model', 'model': m}

# Predict detailed LOB
hier_pred = []
for i, g in enumerate(group_pred):
    info = group_detail_models.get(g)
    if info is None:
        hier_pred.append(global_default_lob)
    elif info['type'] == 'constant':
        hier_pred.append(info['label'])
    else:
        hier_pred.append(info['model'].predict(X_test[i])[0])

hier_acc = accuracy_score(y_test_reset, hier_pred)
hier_f1_macro = f1_score(y_test_reset, hier_pred, average='macro')
hier_f1_weighted = f1_score(y_test_reset, hier_pred, average='weighted')

hierarchical_metrics = pd.DataFrame([{
    'model': 'Hierarchical NB->NB',
    'accuracy': hier_acc,
    'f1_macro': hier_f1_macro,
    'f1_weighted': hier_f1_weighted
}])

hierarchical_metrics



,model,accuracy,f1_macro,f1_weighted
0,Hierarchical NB->NB,0.638772,0.37155,0.619691


## 12) Hybrid Rule + ML Policy

Decision order:
1. strong prefix rule -> auto by rule,
2. else high-confidence ML -> auto by ML,
3. else -> manual review.


In [47]:

# Build rule stats on ML train split for fair combination
train_prefix = ml_df_model.loc[X_train_text.index, 'prefix'].astype(str).reset_index(drop=True)
test_prefix = ml_df_model.loc[X_test_text.index, 'prefix'].astype(str).reset_index(drop=True)

tmp_rule = pd.DataFrame({'prefix': train_prefix, 'lob': y_train_reset})
rule_support = tmp_rule.groupby('prefix').size().rename('support')
rule_top = tmp_rule.groupby('prefix')['lob'].agg(lambda s: s.value_counts().index[0]).rename('rule_lob')
rule_top_count = tmp_rule.groupby('prefix')['lob'].agg(lambda s: s.value_counts().iloc[0]).rename('top_count')

rule_table = pd.concat([rule_support, rule_top, rule_top_count], axis=1).reset_index()
rule_table['confidence'] = rule_table['top_count'] / rule_table['support']

rule_map = dict(zip(rule_table['prefix'], rule_table['rule_lob']))
rule_conf_map = dict(zip(rule_table['prefix'], rule_table['confidence']))
rule_support_map = dict(zip(rule_table['prefix'], rule_table['support']))

best_model_name = ml_metrics.iloc[0]['model']
best_pred = pd.Series(predictions[best_model_name]).reset_index(drop=True)
best_proba = probabilities[best_model_name]
best_classes = trained_models[best_model_name].classes_
max_proba = best_proba.max(axis=1)

rule_pred = test_prefix.map(rule_map)
rule_conf = test_prefix.map(rule_conf_map).fillna(0.0)
rule_sup = test_prefix.map(rule_support_map).fillna(0)

# Hybrid thresholds
RULE_MIN_SUPPORT = 20
RULE_MIN_CONF = 0.90
ML_MIN_CONF = 0.85

use_rule = rule_pred.notna() & (rule_sup >= RULE_MIN_SUPPORT) & (rule_conf >= RULE_MIN_CONF)
use_ml = (~use_rule) & (max_proba >= ML_MIN_CONF)

hybrid_decision = np.where(use_rule, 'auto_rule', np.where(use_ml, 'auto_ml', 'manual_review'))
hybrid_pred = np.where(use_rule, rule_pred, np.where(use_ml, best_pred, None))

hybrid_df = pd.DataFrame({
    'y_true': y_test_reset,
    'prefix': test_prefix,
    'rule_pred': rule_pred,
    'rule_conf': rule_conf,
    'rule_support': rule_sup,
    'ml_pred': best_pred,
    'ml_max_proba': max_proba,
    'decision': hybrid_decision,
    'hybrid_pred': hybrid_pred,
})

auto_mask = hybrid_df['decision'].isin(['auto_rule', 'auto_ml']) & hybrid_df['hybrid_pred'].notna()
manual_mask = hybrid_df['decision'] == 'manual_review'

auto_acc = accuracy_score(hybrid_df.loc[auto_mask, 'y_true'], hybrid_df.loc[auto_mask, 'hybrid_pred']) if auto_mask.any() else np.nan
auto_f1_macro = f1_score(hybrid_df.loc[auto_mask, 'y_true'], hybrid_df.loc[auto_mask, 'hybrid_pred'], average='macro') if auto_mask.any() else np.nan

# top-k quality for ML on full test
top3 = top_k_accuracy_score(y_test_reset, best_proba, k=3, labels=best_classes)
top5 = top_k_accuracy_score(y_test_reset, best_proba, k=5, labels=best_classes)

hybrid_summary = pd.DataFrame([
    {
        'best_ml_model': best_model_name,
        'rule_thresholds': f'support>={RULE_MIN_SUPPORT}, conf>={RULE_MIN_CONF}',
        'ml_threshold': ML_MIN_CONF,
        'auto_share': auto_mask.mean(),
        'manual_share': manual_mask.mean(),
        'auto_accuracy': auto_acc,
        'auto_f1_macro': auto_f1_macro,
        'ml_top3_acc': top3,
        'ml_top5_acc': top5,
    }
])

hybrid_summary



,best_ml_model,rule_thresholds,ml_threshold,auto_share,manual_share,auto_accuracy,auto_f1_macro,ml_top3_acc,ml_top5_acc
0,KNN,"support>=20, conf>=0.9",0.85,0.393906,0.606094,0.878924,0.669748,0.814749,0.840804


## 13) Operational KPI Table (Auto Coverage / Manual Review / Top-3)


In [48]:

# Compare all approaches including hierarchical
all_compare = pd.concat([
    compare_df[['model', 'accuracy', 'f1_macro', 'f1_weighted']],
    hierarchical_metrics[['model', 'accuracy', 'f1_macro', 'f1_weighted']]
], ignore_index=True)

all_compare = all_compare.sort_values('f1_macro', ascending=False).reset_index(drop=True)
print('Model leaderboard (LOB):')
display(all_compare)

print('Hybrid policy KPI:')
display(hybrid_summary)

print('Decision split:')
display(hybrid_df['decision'].value_counts(dropna=False).rename_axis('decision').reset_index(name='count'))



Model leaderboard (LOB):


,model,accuracy,f1_macro,f1_weighted
0,KNN,0.649371,0.463693,0.644658
1,Prefix Rule Baseline,0.556193,0.441478,0.576102
2,Naive Bayes,0.654228,0.372531,0.634601
3,Hierarchical NB->NB,0.638772,0.371550,0.619691
4,Decision Tree,0.177081,0.281057,0.186321


Hybrid policy KPI:


,best_ml_model,rule_thresholds,ml_threshold,auto_share,manual_share,auto_accuracy,auto_f1_macro,ml_top3_acc,ml_top5_acc
0,KNN,"support>=20, conf>=0.9",0.85,0.393906,0.606094,0.878924,0.669748,0.814749,0.840804


Decision split:


,decision,count
0,manual_review,2745
1,auto_ml,1574
2,auto_rule,210
